# Notebook 1 — Basic Linear Regression

This notebook reads the banking CSV, selects one feature, splits the data,
trains a linear regression model, predicts one record and the complete test
set, and displays the R² score.

## Use case

A bank wants to estimate the **balance after a transaction**. The target is
`Balance_After_Transaction`. The simplest model uses the transaction `Amount`
as its input.

Linear regression learns a straight-line relationship:

\[\text{Predicted Balance} = \text{Intercept} + (\text{Coefficient} \times \text{Amount})\]

> **Important:** Regression does not have classification accuracy. `model.score()`
returns **R²**, which measures how much variation in the target is explained by
the model. A high training score alone does not guarantee good performance on
new data.

## 1. Import the required libraries

- **pandas** reads and displays the data.
- **train_test_split** separates training data from unseen test data.
- **LinearRegression** builds the prediction model.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

## 2. Read the dataset

The CSV is read directly from the current folder. `head()` displays the first
five records, while `shape` shows the number of rows and columns.

In [ ]:
df = pd.read_csv("banking_operations(1).csv")
print("Dataset shape:", df.shape)
df.head()

## 3. Select the feature and target

`X` must be two-dimensional, so the feature is selected using double brackets.
`y` contains the numeric value that the model must predict.

In [ ]:
X = df[["Amount"]]
y = df["Balance_After_Transaction"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

## 4. Split the data

80% of the records are used for training and 20% are held back for testing.
`random_state=42` makes the split reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training records:", len(X_train))
print("Testing records :", len(X_test))

## 5. Train the model

`fit()` learns the intercept and coefficient from the training records only.

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print(f"Intercept : {model.intercept_:,.2f}")
print(f"Coefficient for Amount: {model.coef_[0]:,.4f}")

## 6. Predict one new record

Here the model estimates the balance for a transaction amount of ₹15,000.
The column name is kept identical to the training feature.

In [ ]:
single_record = pd.DataFrame({"Amount": [15000]})
single_prediction = model.predict(single_record)[0]

print("Transaction amount:", f"₹{single_record.loc[0, 'Amount']:,.2f}")
print("Predicted balance :", f"₹{single_prediction:,.2f}")

## 7. Predict the test data

These records were not used to train the model. Comparing actual and predicted
values gives a more realistic view of performance.

In [ ]:
test_predictions = model.predict(X_test)

results = pd.DataFrame({
    "Amount": X_test["Amount"].values,
    "Actual_Balance": y_test.values,
    "Predicted_Balance": test_predictions,
})
results["Prediction_Error"] = results["Actual_Balance"] - results["Predicted_Balance"]
results.round(2)

## 8. Display the model score

For linear regression, `score()` returns R². R² = 1 is a perfect fit; R² = 0
means the model is no better than predicting the test-set mean; a negative
value means it performs worse than that baseline.

In [ ]:
train_r2 = model.score(X_train, y_train)
test_r2 = model.score(X_test, y_test)

print(f"Training R² score: {train_r2:.4f}")
print(f"Testing R² score : {test_r2:.4f}")
print(f"Testing R² as a percentage-style display: {test_r2 * 100:.2f}%")

## Conclusion

The model demonstrates the complete basic workflow. Its test R² should be
interpreted together with the small dataset size: only ten records are in the
test set, and transaction amount alone may not explain every balance change.